In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, curve_fit
from emcee import EnsembleSampler

import matplotlib.pyplot as plt

In [2]:
#read in data
df = pd.read_csv("polarizer_correlation.csv")
df

,Trial Number,Angle A,Angle B,N_A,N_B,N_COIN,TIME [s]
0,1.0,0,0.0,NaN,NaN,NaN,NaN
1,2.0,0,22.5,NaN,NaN,NaN,NaN
2,3.0,0,30.0,NaN,NaN,NaN,NaN
3,4.0,0,45.0,NaN,NaN,NaN,NaN
4,5.0,0,60.0,NaN,NaN,NaN,NaN
5,6.0,0,67.5,NaN,NaN,NaN,NaN
6,7.0,0,90.0,NaN,NaN,NaN,NaN
7,8.0,0,100.0,NaN,NaN,NaN,NaN
8,9.0,0,112.5,NaN,NaN,NaN,NaN
9,10.0,0,120.0,NaN,NaN,NaN,NaN


In [ ]:
# clean up plot 

In [ ]:
#plot settings
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath}"
})
plt.rcParams["font.size"] = 15

In [ ]:
def step_model(x, center, width, steepness, amplitude, noise):
    x = np.array(x)
    y = amplitude * np.exp(-((np.abs(x - center) / width)**steepness)) + noise
    return y

In [ ]:
def loss(theta):
    y_mod = step_model(df["Actual Tau?"], *theta)
    y_data = df["Actual Counts"]
    return np.sum((y_data - y_mod)**2/y_data)

In [ ]:
opt_res = minimize(lambda p: loss(np.exp(p)), np.log(p0), method="Nelder-Mead")
tau_fit = np.exp(opt_res.x)[0]
width_fit = np.exp(opt_res.x)[1]

In [ ]:
fig, ax = plt.subplots(dpi = 300)

x = np.linspace(-70, 70, 5000)
y = step_model(x, *np.exp(opt_res.x))
ax.plot(x, y, color = "orange")
ax.errorbar(df['Actual Tau?'], df['Actual Counts'], np.sqrt(df['Actual Counts']), ls="", marker = ".")
ax.set_xlabel("$\\tau$ [ns]")
ax.set_ylabel("Count Rate $[\\text{s}^{-1}]$")
ax.axvline(tau_fit, ls = "dashed", color = "red")
ax.axvline(tau_fit + width_fit, ls = "dotted", color = "red")
ax.axvline(tau_fit - width_fit, ls = "dotted", color = "red")
ax.set_xlim(tau_fit - width_fit - 15, tau_fit + width_fit + 15)
plt.savefig("tau_fit.png")

In [ ]:
#mcmc prior
def log_prior(theta):
    if np.any(theta < 0):
        return -np.inf
    
    return 0

#mcmc log likelihood
def log_prob(theta):
    if log_prior(theta) == -np.inf:
        return -np.inf
    
    return -loss(theta)

In [ ]:
#initialize mcmc walkers
N_WALKERS = 16
BUMP_SIZE = 1
x0 = np.exp(opt_res.x) + BUMP_SIZE * np.random.randn(N_WALKERS, len(opt_res.x))

In [ ]:
#run mcmc
sampler = EnsembleSampler(N_WALKERS, x0.shape[1], log_prob)
sampler.run_mcmc(x0, 25_000, progress=True)

In [ ]:
#names of parameters
param_labels = ["Center",
                "Width",
                "Steepness",
                "Amplitude",
                "Noise"]

In [ ]:
#check fit stability 
autocorr_time = sampler.get_autocorr_time()
chain = sampler.get_chain(discard=int(np.max(autocorr_time*4)))
for dim_n in range(chain.shape[2]):
    ax = plt.subplot(chain.shape[2], 1, dim_n + 1)
    ax.set_ylabel(param_labels[dim_n])
    for walker_n in range(chain.shape[1]):
        plt.plot(chain[:, walker_n, dim_n], c='black', alpha=0.1)

In [ ]:
#examine posterior distributions of parameters
import corner
corner.corner(sampler.get_chain(discard=int(np.max(autocorr_time*4)), flat=True), labels=param_labels);

In [ ]:
chain = sampler.get_chain(flat=True, discard=int(np.max(autocorr_time*4)))
results = pd.DataFrame(
    {
        "Median Value": np.median(chain, axis = 0),
        "2-sigma Uncertainty": np.sqrt(np.diag(np.cov(chain.T))) * 2
    },
    index=param_labels
)
results["Uncertainty/Value"] = results["2-sigma Uncertainty"]/results["Median Value"]

In [ ]:
results